# Toy SCF Convergence

This notebook runs the first spin-unpolarized Γ-point SCF loop. The model is intentionally tiny: one smooth local pseudopotential, two electrons in one doubly occupied orbital, Hartree potential, and Dirac LDA exchange. The purpose is to inspect convergence behavior, not to predict a real material.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.dft import LocalGaussianPseudopotential, RealSpaceGrid, SCFConfig, run_scf

The automatic solver uses the dense Hamiltonian path for very small grids and the FFT-based imaginary-time path above that threshold. Here we force the larger-grid path so the notebook remains quick while still exercising the plane-wave operators.

In [ ]:
grid = RealSpaceGrid((8, 8, 8), [8.0, 8.0, 8.0])
local = LocalGaussianPseudopotential(
    centers=[[4.0, 4.0, 4.0]],
    amplitudes=-3.0,
    widths=0.9,
)
config = SCFConfig(
    max_iterations=12,
    tolerance=1e-6,
    mixing=0.4,
    solver="auto",
    max_dense_grid_points=256,
    seed=11,
)

result = run_scf(grid, local, electron_count=2.0, config=config)
result.to_dict()

The residual measures how much the density changes between SCF iterations. For this first milestone, monotonic convergence is not guaranteed; the trace is mainly a diagnostic for mixing and solver choices.

In [ ]:
history = result.history
iterations = [row["iteration"] for row in history]
residual = [row["residual"] for row in history]
energy = [row["total"] for row in history]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].semilogy(iterations, residual, marker="o")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("density residual")
axes[0].set_title("SCF residual")

axes[1].plot(iterations, energy, marker="o")
axes[1].set_xlabel("iteration")
axes[1].set_ylabel("energy / hartree")
axes[1].set_title("total energy trace");

The final density is a normal grid array. This is the surface we will use for later visual diagnostics and, eventually, forces and geometry updates.

In [ ]:
mid = grid.shape[2] // 2
plt.figure(figsize=(4.5, 4))
plt.imshow(np.array(result.density)[:, :, mid], origin="lower", cmap="magma")
plt.title("final density slice")
plt.colorbar(label="ρ(r)");